# পাঠ ১৬ - Microsoft Foundry এর সাথে স্কেলযোগ্য এজেন্ট ডিপ্লয়মেন্ট

এই নোটবুকটিতে আপনি একটি কাল্পনিক কোম্পানি **Contoso** এর জন্য একটি **প্রোডাকশন-রেডি কাস্টমার সাপোর্ট এজেন্ট** তৈরি করবেন। আগের পাঠগুলির বিপরীতে, এখানে মূল বিষয়টি হলো এজেন্টের যুক্তি চক্র নয় — বরং তার চারপাশের সব কিছু যা একটি এজেন্টকে স্কেলে নিরাপদে চালানোর যোগ্য করে তোলে:

১. **টুল কলিং** — অর্ডার অনুসন্ধান এবং টিকিট তৈরি।
২. **RAG** — একটি জ্ঞানভিত্তিক নীতিমালা অনুযায়ী উত্তর প্রদান।
৩. **মেমোরি** — টার্নগুলির মধ্যে গ্রাহককে মনে রাখা।
৪. **মডেল রাউটিং** — সাধারণ অনুরোধ সামান্য মডেলে পাঠানো, জটিল অনুরোধ বড় মডেলে পাঠানো।
৫. **প্রতিক্রিয়া ক্যাশিং** — পুনরাবৃত্ত প্রশ্নগুলির জন্য মডেল কল ছাড়াই উত্তর প্রদান।
৬. **ম্যানুয়াল অনুমোদন** — একটি নির্ধারিত সীমার উপরে অর্থ ফেরতের জন্য স্বাক্ষরের জন্য বিরতি।
৭. **মূল্যায়ন গেট** — একটি অফলাইন টেস্ট সেট যা খারাপ রিলিজকে ব্লক করে।
৮. **অবজারভেবিলিটি** — প্রতিটি অনুরোধের চারপাশে OpenTelemetry ট্রেসিং।

প্রতিটি বিভাগ স্বয়ংসম্পূর্ণ এবং রানের উপযোগী। প্রতিটি লাইন পড়ুন — প্রোডাকশন প্রিমিটিভগুলি ইচ্ছাকৃতভাবে ছোট রাখা হয়েছে।


## সেটআপ

এই নোটবুক চালানোর আগে, নিশ্চিত করুন আপনার কাছে:

১. **একটি Microsoft Foundry প্রকল্প** যার সাথে একটি ডিপ্লয় করা চ্যাট মডেল আছে (যেমন `gpt-5-mini`)।
২. **Azure CLI দিয়ে লগ ইন করেছেন** — আপনার টার্মিনালে `az login` চালান।
৩. **প্রয়োজনীয় পরিবেশ ভেরিয়েবলগুলি সেট করেছেন:**
   - `AZURE_AI_PROJECT_ENDPOINT` — আপনার Microsoft Foundry প্রকল্পের এন্ডপয়েন্ট।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার ডিপ্লয় করা মডেলের নাম।

RAG সেকশনটি **Azure AI Search** ব্যবহার করে যখন `AZURE_SEARCH_SERVICE_ENDPOINT` এবং `AZURE_SEARCH_API_KEY` সেট থাকে, এবং না থাকলে একটি ইন-মেমরি সার্চ ব্যবহৃত হয় যাতে নোটবুকটি সার্চ রিসোর্স ছাড়াই চালাতে পারে।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. সরঞ্জাম

উৎপাদন সরঞ্জামগুলি বাস্তব সিস্টেমের বিরুদ্ধে বাস্তব কাজ করে। এখানে আমরা সাধারণ পাইথন ফাংশন দিয়ে একটি অর্ডার ডেটাবেস এবং একটি টিকেটিং সিস্টেম অনুকরণ করি। `@tool` ডেকোরেটর এগুলোকে এজেন্টের কাছে উন্মুক্ত করে।

লক্ষ্য করুন `issue_refund` থ্রেশহোল্ড এর উপরে রিফান্ডের জন্য `approval_mode="always_require"` ব্যবহার করে — এটি হলো মানব-অন্তর্ভুক্ত প্রিমিটিভ যা আমরা পরে প্রয়োগ করব।


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — নীতিমালা জ্ঞান আধার

নীতিমালা প্রশ্ন ("আপনার রিটার্ন উইন্ডো কত?") একটি কর্তৃত্বপূর্ণ সূত্র থেকে উত্তর দেয়া উচিত, মডেলের স্মৃতি থেকে নয়। আমরা একটি ছোট জ্ঞান আধারকে একটি অনুসন্ধান সরঞ্জাম হিসেবে মোড়া করি।

উৎপাদনে এটি হল **Azure AI Search**; এখানে আমরা একটি ইন-মেমরি কীওয়ার্ড অনুসন্ধান প্রদান করি যাতে নোটবুক যেকোনো জায়গায় চলতে পারে, এবং পরিবেশ ভেরিয়েবল উপস্থিত থাকলে স্বয়ংক্রিয়ভাবে Azure AI Search-এ সুইচ করি।


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## ৩. মেমোরি

একটি সাপোর্ট এজেন্ট যে কার সাথে কথা বলছে তা ভুলে যায়, সে একটি খারাপ সাপোর্ট এজেন্ট। আমরা প্রতিটি গ্রাহকের জন্য একটি ছোট প্রোফাইল স্টোর রাখি এবং এজেন্টের নির্দেশাবলীতে একটি সংক্ষিপ্ত সারাংশ সংযোজন করি। প্রোডাকশনে এটি একটি মেমোরি সার্ভিস (দেখুন লেসন ১৩); এখানে একটি ডিকশনারি প্যাটার্নটি দৃশ্যমান করে তোলে।


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## ৪ ও ৫। মডেল রাউটিং এবং প্রতিক্রিয়া ক্যাশিং

একটি একক অনুরোধ হ্যান্ডলারের সাথে সংযুক্ত দুটি খরচ লিভার:

- **রাউটিং**: একটি সস্তা হিউরিস্টিক ক্লাসিফায়ার নির্ধারণ করে যে একটি অনুরোধ ছোট নাকি বড় মডেলের প্রয়োজন।
- **ক্যাশিং**: স্বাভাবিককৃত পুনরাবৃত্ত প্রশ্নগুলি কোন মডেল কল ছাড়াই সরাসরি ক্যাশ থেকে পরিবেশন করা হয়।

ক্লাসিফায়ারটি এখানে ইচ্ছাকৃতভাবে সহজ। প্রোডাকশনে আপনি এটি ট্রাফিকের বিরুদ্ধে যাচাই করবেন এবং এটি Foundry এর Model Router দিয়ে প্রতিস্থাপন করতে পারেন।


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## ৬ এবং ৮। এজেন্ট, মানব অনুমোদন, এবং পর্যবেক্ষণযোগ্যতা

এখন আমরা উপরের টুলগুলো থেকে এজেন্ট তৈরি করি এবং প্রতিটি অনুরোধকে একটি OpenTelemetry স্প্যানের মধ্যে আবৃত করি। `handle_support_request` ফাংশনটি প্রোডাকশন অনুরোধ হ্যান্ডলার: ক্যাশ → রুট → ট্রেস → রান → ক্যাশ।

মানব অনুমোদন ফ্রেমওয়ার্ক দ্বারা পরিচালিত হয়: কারণ `issue_refund` এর `approval_mode="always_require"`, রান থেমে যায় এবং একটি অনুমোদন অনুরোধ উপস্থাপন করে যা আমরা স্পষ্টভাবে সমাধান করি।


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## ৭. মূল্যায়ন গেট

এইটি পাঠের রিলিজ গেট: একটি অফলাইন টেস্ট সেট এজেন্টকে স্কোর করে, এবং ডিপ্লয়মেন্ট তখনই এগোয় যখন পাস রেট একটি নির্দিষ্ট সীমা অতিক্রম করে। এখানে স্কোরারটি একটি সাধারণ কীওয়ার্ড-ওভারল্যাপ চেক যা নোটবুককে স্বনির্ভর রাখে; প্রোডাকশনে আপনি একটি LLM-অ্যাস-জাজ বা একটি ফ্রেমওয়ার্ক ইভ্যালুয়েটর ব্যবহার করবেন (দেখুন পাঠ ১০)।


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## একসাথে করা: একটি সিমুলেটেড রিলিজ

নিচের সেলটি পুরো লুপটি দেখায় যা পাঠটি বর্ণনা করে: ইভালুয়েশন গেট চালান, এবং শুধুমাত্র যদি এটি উত্তীর্ণ হয় তখনই "ডিপ্লয়" করুন। এটি সেই প্যাটার্ন যা আপনি Foundry Agent Service-এ একটি এজেন্ট সংস্করণ প্রমোট করার আগে CI-তে চালাবেন।


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## সারসংক্ষেপ

আপনি একটি উৎপাদন-সিদ্ধ গ্রাহক সহায়তা এজেন্ট তৈরি করেছেন, যার প্রতিটি অপারেশনাল দিক সংযুক্ত:

- **টুলস, RAG, এবং মেমোরি** এজেন্টকে ক্ষমতা এবং প্রেক্ষাপট দেয়।
- **মডেল রাউটিং এবং ক্যাশিং** বিলম্ব এবং খরচ নিয়ন্ত্রণে রাখে।
- **মানব অনুমোদন** বড় রিফান্ডের মত উচ্চ-ঝুঁকিপূর্ণ কার্যক্রম রক্ষা করে।
- **মূল্যায়ন গেট** খারাপ রিলিজগুলো বাজারে আসার আগে বাধা দেয়।
- **ট্রেসিং** প্রতিটি অনুরোধকে পর্যবেক্ষণযোগ্য করে তোলে।

### চ্যালেঞ্জ

এই এজেন্টটিকে সম্প্রসারিত করুন:

১. **একাধিক মডেল সমর্থন করুন** — তৃতীয় "যুক্তি" স্তর যোগ করুন এবং স্কেলেশন/অভিযোগগুলো সেখানে পাঠান।
২. **মূল্যায়ন গেট যোগ করুন** — `TEST_CASES` সম্প্রসারিত করে রিফান্ড-অনুমোদন পরিস্থিতি যুক্ত করুন এবং নিশ্চিত করুন গেটটি রিগ্রেশন আটকায়।
৩. **খরচ সচেতন রাউটিং যোগ করুন** — প্রতিটি অনুরোধের আনুমানিক খরচ (ছোট, বড়, ক্যাশ) ট্র্যাক করুন এবং মিশ্র প্রশ্নগুলোর ব্যাচ শেষে খরচ রিপোর্ট প্রিন্ট করুন।

পরবর্তী পাঠে আপনি বিপরীত যাত্রা করবেন এবং Microsoft Foundry Local ও Qwen ব্যবহার করে সম্পূর্ণ আপনার নিজস্ব মেশিনে এজেন্ট চালাবেন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
